In [ ]:
import os, numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as pre
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import seaborn as sns

BASE = '/kaggle/input/datasets/ekrasafdar/brain-tumor-mri'
TRAIN_DIR = os.path.join(BASE, 'Training')
TEST_DIR  = os.path.join(BASE, 'Testing')

# Build MC Dropout model
tf.random.set_seed(42); np.random.seed(42)
inp = layers.Input(shape=(224, 224, 3))
base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inp)
base.trainable = True
for L in base.layers[:-50]: L.trainable = False
x = layers.GlobalAveragePooling2D()(base.output)
x = layers.BatchNormalization()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x, training=True)  # MC Dropout
out = layers.Dense(4, activation='softmax')(x)
model = keras.Model(inp, out)
model.compile(optimizer=keras.optimizers.Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

# Data
tr_aug = ImageDataGenerator(preprocessing_function=pre, rotation_range=20,
    width_shift_range=0.15, height_shift_range=0.15, zoom_range=0.15,
    horizontal_flip=True, brightness_range=[0.8, 1.2], validation_split=0.2)
tr = tr_aug.flow_from_directory(TRAIN_DIR, target_size=(224,224), batch_size=32,
    class_mode='categorical', subset='training', seed=42)
val = tr_aug.flow_from_directory(TRAIN_DIR, target_size=(224,224), batch_size=32,
    class_mode='categorical', subset='validation', shuffle=False, seed=42)
te = ImageDataGenerator(preprocessing_function=pre).flow_from_directory(
    TEST_DIR, target_size=(224,224), batch_size=32, class_mode='categorical', shuffle=False)

# Train
cb = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
]
model.fit(tr, validation_data=val, epochs=30, callbacks=cb, verbose=1)

# Eval
te.reset(); loss, acc = model.evaluate(te, verbose=0)
te.reset(); yp = np.argmax(model.predict(te, verbose=0), axis=1)
yt = te.classes
print(f"\nMC Model — Accuracy: {acc*100:.2f}% | F1: {f1_score(yt, yp, average='weighted')*100:.2f}%")

# Uncertainty (MC Dropout inference)
print("\nRunning MC Dropout uncertainty...")
te.reset()
all_preds = []
for _ in range(50):
    te.reset()
    all_preds.append(model.predict(te, verbose=0))
all_preds = np.array(all_preds)
mean_pred = np.mean(all_preds, axis=0)
entropy = -np.sum(mean_pred * np.log(mean_pred + 1e-10), axis=1)
y_true = te.classes
y_pred = np.argmax(mean_pred, axis=1)
correct = (y_true == y_pred)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(entropy[correct], bins=25, alpha=0.7, label='Correct', color='green', density=True)
axes[0].hist(entropy[~correct], bins=25, alpha=0.7, label='Incorrect', color='red', density=True)
axes[0].set_xlabel('Predictive Entropy'); axes[0].set_ylabel('Density')
axes[0].set_title('Correct vs Incorrect'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']
class_entropy = [entropy[y_true == i] for i in range(4)]
bp = axes[1].boxplot(class_entropy, labels=class_names, patch_artist=True)
for patch, color in zip(bp['boxes'], ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[1].set_ylabel('Predictive Entropy'); axes[1].set_title('Per-Class Uncertainty')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/uncertainty_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Stats
print(f"\nMean entropy (correct): {np.mean(entropy[correct]):.4f}")
print(f"Mean entropy (incorrect): {np.mean(entropy[~correct]):.4f}")
for i, name in enumerate(class_names):
    print(f"{name}: {np.mean(entropy[y_true == i]):.4f}")

print("\n✅ DONE. Download uncertainty_analysis.png")